# 03 - Baseline for submission

## 1. Load data

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

ROOT = Path("..")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
OUTPUTS = ROOT / "outputs"
SUB_DIR = OUTPUTS / "submissions"
CV_DIR = OUTPUTS / "cv_results"

SUB_DIR.mkdir(parents=True, exist_ok=True)
CV_DIR.mkdir(parents=True, exist_ok=True)

daily_panel = pd.read_parquet(DATA_PROCESSED / "daily_panel.parquet")
sku_activity = pd.read_parquet(DATA_PROCESSED / "sku_activity.parquet")
sample = pd.read_csv(DATA_RAW / "sample_submission.csv")

daily_panel["Date"] = pd.to_datetime(daily_panel["Date"])

for col in [
    "first_sale_date", "last_sale_date",
    "first_transaction_date", "last_transaction_date"
]:
    if col in sku_activity.columns:
        sku_activity[col] = pd.to_datetime(sku_activity[col])

print("daily_panel:", daily_panel.shape)
print("sku_activity:", sku_activity.shape)
print("sample:", sample.shape)

display(daily_panel.head())
display(sku_activity.head())
display(sample.head())

daily_panel: (28014888, 17)
sku_activity: (15972, 20)
sample: (31944, 29)


,ItemCode,Date,y_net,y_gross,y_return,sales,cost,profit,transaction_count,y_net_clip,dayofweek,is_saturday,is_sunday,month,day,weekofyear,year
0,SKU-00001,2020-11-17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0,11,17,47,2020
1,SKU-00001,2020-11-18,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,0,0,11,18,47,2020
2,SKU-00001,2020-11-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,0,0,11,19,47,2020
3,SKU-00001,2020-11-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0,0,11,20,47,2020
4,SKU-00001,2020-11-21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,1,0,11,21,47,2020


,ItemCode,active_days,active_net_days,total_y_net,total_y_gross,total_return,total_sales,total_cost,total_profit,total_transactions,first_sale_date,last_sale_date,first_transaction_date,last_transaction_date,has_ever_sold,days_since_last_sale,days_since_last_transaction,positive_profit,profit_rank,return_rate_qty
0,SKU-00001,15,15,30.0,30.0,0.0,3.608433e+07,0.0,3.608433e+07,30.0,2025-05-26,2025-08-28,2025-05-26,2025-08-28,1,8,8,3.608433e+07,781,0.0
1,SKU-00002,895,895,5894.0,5894.0,0.0,8.012686e+09,0.0,8.012686e+09,5896.0,2022-01-10,2025-09-05,2022-01-10,2025-09-05,1,0,0,8.012686e+09,2,0.0
2,SKU-00003,1061,1061,10935.0,10935.0,0.0,1.670068e+10,0.0,1.670068e+10,10936.0,2022-01-03,2025-09-04,2022-01-03,2025-09-04,1,1,1,1.670068e+10,1,0.0
3,SKU-00004,279,279,659.0,659.0,0.0,8.359968e+08,0.0,8.359968e+08,659.0,2023-07-19,2024-12-20,2023-07-19,2024-12-20,1,259,259,8.359968e+08,12,0.0
4,SKU-00005,327,327,1101.0,1101.0,0.0,2.243340e+09,0.0,2.243340e+09,1103.0,2022-01-03,2023-06-26,2022-01-03,2023-06-26,1,802,802,2.243340e+09,4,0.0


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,SKU-00002_validation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,SKU-00003_validation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,SKU-00004_validation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,SKU-00005_validation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [3]:
print("Date range:", daily_panel["Date"].min(), "→", daily_panel["Date"].max())
print("Number of SKUs:", daily_panel["ItemCode"].nunique())
print("Sample rows:", len(sample))
print("Sample IDs unique:", sample["id"].nunique())

assert len(sample) == sample["id"].nunique()

Date range: 2020-11-17 00:00:00 → 2025-09-05 00:00:00
Number of SKUs: 15972
Sample rows: 31944
Sample IDs unique: 31944


## 2. Config last forecast

In [4]:
FINAL_TRAIN_END = pd.Timestamp("2025-09-05")
FINAL_FORECAST_START = pd.Timestamp("2025-09-06")
HORIZON = 56

TARGET_FOR_PRED = "y_net_clip"  # prediction không được âm
TARGET_FOR_METRIC = "y_net"     # local WRMSSE đang dùng y_net

forecast_dates = pd.date_range(FINAL_FORECAST_START, periods=HORIZON, freq="D")

print(forecast_dates[0], "→", forecast_dates[-1])
print("Day of week start:", forecast_dates[0].day_name())

2025-09-06 00:00:00 → 2025-10-31 00:00:00
Day of week start: Saturday


## 3. Baseline component 1: recent mean

In [6]:
def make_recent_blend_prediction(
    panel: pd.DataFrame,
    train_end,
    forecast_start,
    horizon: int = 56,
    target_col: str = "y_net_clip",
    windows=(28, 56, 112),
    weights=(0.50, 0.30, 0.20)
) -> pd.DataFrame:
    """
    Predict each future day by blended recent mean per SKU.
    """
    train_end = pd.Timestamp(train_end)
    forecast_start = pd.Timestamp(forecast_start)
    forecast_dates = pd.date_range(forecast_start, periods=horizon, freq="D")
    
    itemcodes = sorted(panel["ItemCode"].unique())
    
    assert len(windows) == len(weights)
    assert abs(sum(weights) - 1.0) < 1e-9
    
    blended = pd.Series(0.0, index=itemcodes)
    
    for window, weight in zip(windows, weights):
        hist_start = train_end - pd.Timedelta(days=window - 1)
        
        hist = panel.loc[
            (panel["Date"] >= hist_start) &
            (panel["Date"] <= train_end),
            ["ItemCode", target_col]
        ]
        
        mean_by_sku = hist.groupby("ItemCode")[target_col].mean()
        blended += weight * mean_by_sku.reindex(itemcodes).fillna(0)
    
    pred = pd.DataFrame(
        np.repeat(blended.to_numpy()[:, None], horizon, axis=1),
        index=itemcodes,
        columns=forecast_dates
    )
    
    return pred

## 4. Baseline compononent 2: same day-of-week mean

In [7]:
def make_same_dow_prediction(
    panel: pd.DataFrame,
    train_end,
    forecast_start,
    horizon: int = 56,
    target_col: str = "y_net_clip",
    window_days: int = 182
) -> pd.DataFrame:
    """
    For each future date, predict using SKU's historical mean on the same day-of-week
    within the recent window.
    """
    train_end = pd.Timestamp(train_end)
    forecast_start = pd.Timestamp(forecast_start)
    forecast_dates = pd.date_range(forecast_start, periods=horizon, freq="D")
    
    hist_start = train_end - pd.Timedelta(days=window_days - 1)
    itemcodes = sorted(panel["ItemCode"].unique())
    
    hist = panel.loc[
        (panel["Date"] >= hist_start) &
        (panel["Date"] <= train_end),
        ["ItemCode", "dayofweek", target_col]
    ].copy()
    
    dow_mean = (
        hist.groupby(["ItemCode", "dayofweek"])[target_col]
        .mean()
        .unstack("dayofweek")
        .reindex(index=itemcodes, columns=range(7))
        .fillna(0)
    )
    
    pred = pd.DataFrame(index=itemcodes, columns=forecast_dates, dtype=float)
    
    for d in forecast_dates:
        pred[d] = dow_mean[d.dayofweek].to_numpy()
    
    return pred

## 5. Baseline component 3: yearly lag

In [8]:
def make_yearly_lag_prediction(
    panel: pd.DataFrame,
    train_end,
    forecast_start,
    horizon: int = 56,
    target_col: str = "y_net_clip",
    lags=(364, 728)
) -> pd.DataFrame:
    """
    Predict using values from approximately same weekday in previous years.
    lag=364 keeps weekday alignment.
    """
    train_end = pd.Timestamp(train_end)
    forecast_start = pd.Timestamp(forecast_start)
    forecast_dates = pd.date_range(forecast_start, periods=horizon, freq="D")
    
    itemcodes = sorted(panel["ItemCode"].unique())
    
    pred_sum = pd.DataFrame(0.0, index=itemcodes, columns=forecast_dates)
    pred_count = pd.DataFrame(0.0, index=itemcodes, columns=forecast_dates)
    
    needed_hist_dates = []
    for lag in lags:
        needed_hist_dates.extend([d - pd.Timedelta(days=lag) for d in forecast_dates])
    
    needed_hist_dates = sorted(set(needed_hist_dates))
    
    hist = panel.loc[
        panel["Date"].isin(needed_hist_dates),
        ["ItemCode", "Date", target_col]
    ].copy()
    
    hist_wide = (
        hist.pivot(index="ItemCode", columns="Date", values=target_col)
        .reindex(index=itemcodes)
        .fillna(0)
    )
    
    for d in forecast_dates:
        for lag in lags:
            hist_date = d - pd.Timedelta(days=lag)
            
            if hist_date <= train_end and hist_date in hist_wide.columns:
                pred_sum[d] += hist_wide[hist_date].to_numpy()
                pred_count[d] += 1
    
    pred = pred_sum / pred_count.replace(0, np.nan)
    pred = pred.fillna(0)
    
    return pred

## 6. Cap prediction function

In [9]:
def compute_sku_caps(
    panel: pd.DataFrame,
    sku_activity: pd.DataFrame,
    train_end,
    target_col: str = "y_net_clip"
) -> pd.Series:
    """
    Compute SKU-level upper cap for predictions.
    Conservative cap for sparse SKU, looser cap for high-profit SKU.
    """
    train_end = pd.Timestamp(train_end)
    itemcodes = sorted(panel["ItemCode"].unique())
    
    hist_365 = panel.loc[
        (panel["Date"] >= train_end - pd.Timedelta(days=364)) &
        (panel["Date"] <= train_end),
        ["ItemCode", "Date", target_col]
    ].copy()
    
    hist_112 = panel.loc[
        (panel["Date"] >= train_end - pd.Timedelta(days=111)) &
        (panel["Date"] <= train_end),
        ["ItemCode", "Date", target_col]
    ].copy()
    
    hist_56 = panel.loc[
        (panel["Date"] >= train_end - pd.Timedelta(days=55)) &
        (panel["Date"] <= train_end),
        ["ItemCode", "Date", target_col]
    ].copy()
    
    # positive-day quantile, avoids full-calendar quantile becoming 0 for sparse SKU
    positive_hist = hist_365[hist_365[target_col] > 0]
    
    pos_q95 = positive_hist.groupby("ItemCode")[target_col].quantile(0.95)
    max_56 = hist_56.groupby("ItemCode")[target_col].max()
    mean_112 = hist_112.groupby("ItemCode")[target_col].mean()
    
    cap = pd.DataFrame(index=itemcodes)
    cap["pos_q95"] = pos_q95.reindex(itemcodes).fillna(0)
    cap["max_56"] = max_56.reindex(itemcodes).fillna(0)
    cap["mean_112"] = mean_112.reindex(itemcodes).fillna(0)
    
    cap["base_cap"] = np.maximum.reduce([
        cap["pos_q95"] * 2.0,
        cap["max_56"] * 1.2,
        cap["mean_112"] * 5.0
    ])
    
    meta = sku_activity.set_index("ItemCode").reindex(itemcodes)
    
    # Nếu SKU từng bán gần đây, cap tối thiểu là 1 để không bóp quá tay.
    recently_active = meta["days_since_last_sale"].fillna(9999) <= 56
    cap.loc[recently_active, "base_cap"] = cap.loc[recently_active, "base_cap"].clip(lower=1.0)
    
    # Top profit SKU cap rộng hơn, tránh bóp mất spike thật.
    profit_rank = meta["profit_rank"].fillna(999999)
    
    cap_multiplier = pd.Series(1.0, index=itemcodes)
    cap_multiplier.loc[profit_rank <= 100] = 3.0
    cap_multiplier.loc[(profit_rank > 100) & (profit_rank <= 500)] = 2.0
    cap_multiplier.loc[(profit_rank > 500) & (profit_rank <= 1000)] = 1.5
    
    final_cap = cap["base_cap"] * cap_multiplier
    
    return final_cap.fillna(0)

## 7. Post-processing function

In [10]:
def postprocess_prediction(
    pred: pd.DataFrame,
    panel: pd.DataFrame,
    sku_activity: pd.DataFrame,
    train_end,
    sunday_factor: float = 0.0,
    apply_cap: bool = True
) -> pd.DataFrame:
    """
    Apply business/metric-aware rules:
    - non-negative
    - Sunday reduction
    - inactive SKU zero/shrink
    - upper cap
    """
    pred = pred.copy()
    pred[pred < 0] = 0
    
    # 1. Sunday correction
    sunday_cols = [c for c in pred.columns if pd.Timestamp(c).dayofweek == 6]
    if sunday_cols:
        pred.loc[:, sunday_cols] *= sunday_factor
    
    # 2. Inactive rules
    meta = sku_activity.set_index("ItemCode").reindex(pred.index)
    
    days_since = meta["days_since_last_sale"].fillna(9999)
    active_days = meta["active_days"].fillna(0)
    profit_rank = meta["profit_rank"].fillna(999999)
    
    # Zero hard: SKU lâu không bán + không thuộc nhóm profit quan trọng
    zero_mask = (
        ((days_since > 365) & (profit_rank > 500)) |
        ((days_since > 180) & (active_days <= 3) & (profit_rank > 100)) |
        ((days_since > 90) & (active_days <= 1) & (profit_rank > 100))
    )
    
    pred.loc[zero_mask, :] = 0
    
    # Shrink: SKU hơi inactive, nhưng không zero cứng
    shrink_mask = (
        (days_since > 90) &
        (days_since <= 365) &
        (active_days <= 5) &
        (profit_rank > 500) &
        (~zero_mask)
    )
    
    pred.loc[shrink_mask, :] *= 0.25
    
    # 3. Cap
    if apply_cap:
        caps = compute_sku_caps(
            panel=panel,
            sku_activity=sku_activity,
            train_end=train_end,
            target_col=TARGET_FOR_PRED
        ).reindex(pred.index).fillna(0)
        
        pred = pred.clip(upper=caps, axis=0)
    
    pred = pred.fillna(0)
    pred[pred < 0] = 0
    
    return pred

## 8. Build baseline forecast

In [11]:
def make_baseline_v1_forecast(
    panel: pd.DataFrame,
    sku_activity: pd.DataFrame,
    train_end,
    forecast_start,
    horizon: int = 56,
    target_col: str = "y_net_clip",
    sunday_factor: float = 0.0
) -> pd.DataFrame:
    """
    Baseline v1:
    recent blend + same day-of-week + yearly lag + postprocess.
    """
    pred_recent = make_recent_blend_prediction(
        panel,
        train_end=train_end,
        forecast_start=forecast_start,
        horizon=horizon,
        target_col=target_col,
        windows=(28, 56, 112),
        weights=(0.50, 0.30, 0.20)
    )
    
    pred_samedow = make_same_dow_prediction(
        panel,
        train_end=train_end,
        forecast_start=forecast_start,
        horizon=horizon,
        target_col=target_col,
        window_days=182
    )
    
    pred_yearly = make_yearly_lag_prediction(
        panel,
        train_end=train_end,
        forecast_start=forecast_start,
        horizon=horizon,
        target_col=target_col,
        lags=(364, 728)
    )
    
    # Blend chính
    pred = (
        0.45 * pred_recent +
        0.40 * pred_samedow +
        0.15 * pred_yearly
    )
    
    pred = postprocess_prediction(
        pred,
        panel=panel,
        sku_activity=sku_activity,
        train_end=train_end,
        sunday_factor=sunday_factor,
        apply_cap=True
    )
    
    return pred

## 9. Local compute Validation

In [12]:
def compute_sku_metric_info(
    panel: pd.DataFrame,
    train_end,
    target_col: str = "y_net",
    profit_col: str = "profit",
    eps: float = 1e-8
) -> pd.DataFrame:
    """
    Compute SKU-level profit weight and RMSSE scale up to train_end.
    
    weight_i = max(sum(profit_i), 0) / total_positive_profit
    scale_i = mean((y_t - y_{t-1})^2) over training history
    """
    train_end = pd.Timestamp(train_end)
    
    hist = panel.loc[
        panel["Date"] <= train_end,
        ["ItemCode", "Date", target_col, profit_col]
    ].copy()
    
    hist = hist.sort_values(["ItemCode", "Date"])
    
    # 1. Profit weight
    profit_info = (
        hist.groupby("ItemCode", as_index=False)[profit_col]
        .sum()
        .rename(columns={profit_col: "total_profit_metric_train"})
    )
    
    profit_info["positive_profit"] = profit_info["total_profit_metric_train"].clip(lower=0)
    
    total_positive_profit = profit_info["positive_profit"].sum()
    
    if total_positive_profit <= 0:
        raise ValueError("Total positive profit is zero. Cannot compute weights.")
    
    profit_info["weight"] = profit_info["positive_profit"] / total_positive_profit
    
    # 2. RMSSE scale
    hist["diff"] = hist.groupby("ItemCode")[target_col].diff()
    hist["sq_diff"] = hist["diff"] ** 2
    
    scale_info = (
        hist.dropna(subset=["sq_diff"])
        .groupby("ItemCode", as_index=False)["sq_diff"]
        .mean()
        .rename(columns={"sq_diff": "scale"})
    )
    
    metric_info = profit_info.merge(scale_info, on="ItemCode", how="left")
    
    metric_info["scale"] = metric_info["scale"].fillna(0)
    metric_info["scale_safe"] = metric_info["scale"].clip(lower=eps)
    metric_info["zero_scale_flag"] = (metric_info["scale"] < eps).astype(int)
    
    metric_info["profit_rank"] = (
        metric_info["positive_profit"]
        .rank(method="min", ascending=False)
        .astype(int)
    )
    
    metric_info = metric_info.sort_values("profit_rank").reset_index(drop=True)
    metric_info["cum_weight"] = metric_info["weight"].cumsum()
    
    return metric_info

In [13]:
def make_actual_matrix(
    panel: pd.DataFrame,
    start_date,
    horizon: int = 56,
    target_col: str = "y_net"
) -> pd.DataFrame:
    """
    Create actual matrix:
    index = ItemCode
    columns = validation dates
    values = actual target
    """
    start_date = pd.Timestamp(start_date)
    dates = pd.date_range(start_date, periods=horizon, freq="D")
    
    subset = panel.loc[
        panel["Date"].isin(dates),
        ["ItemCode", "Date", target_col]
    ].copy()
    
    actual_wide = (
        subset.pivot(index="ItemCode", columns="Date", values=target_col)
        .sort_index()
        .reindex(columns=dates)
    )
    
    if actual_wide.isna().any().any():
        raise ValueError("Actual matrix contains NaN. Check date range or panel completeness.")
    
    return actual_wide

In [14]:
def wrmsse_score(
    actual_wide: pd.DataFrame,
    pred_wide: pd.DataFrame,
    metric_info: pd.DataFrame,
    clip_pred: bool = True
):
    """
    Compute WRMSSE.
    
    actual_wide:
        index = ItemCode
        columns = forecast dates
    pred_wide:
        same shape as actual_wide
    metric_info:
        must contain ItemCode, weight, scale_safe
    """
    actual_wide = actual_wide.sort_index()
    
    pred_wide = (
        pred_wide
        .reindex(index=actual_wide.index, columns=actual_wide.columns)
        .fillna(0)
        .sort_index()
    )
    
    actual_values = actual_wide.to_numpy(dtype=float)
    pred_values = pred_wide.to_numpy(dtype=float)
    
    if clip_pred:
        pred_values = np.clip(pred_values, 0, None)
    
    mse = np.mean((actual_values - pred_values) ** 2, axis=1)
    
    detail = pd.DataFrame({
        "ItemCode": actual_wide.index,
        "mse": mse
    })
    
    detail = detail.merge(
        metric_info[[
            "ItemCode",
            "weight",
            "scale",
            "scale_safe",
            "total_profit_metric_train",
            "positive_profit",
            "profit_rank"
        ]],
        on="ItemCode",
        how="left"
    )
    
    if detail["weight"].isna().any():
        raise ValueError("Some SKUs in actual_wide are missing from metric_info.")
    
    detail["rmsse"] = np.sqrt(detail["mse"] / detail["scale_safe"])
    detail["weighted_rmsse"] = detail["weight"] * detail["rmsse"]
    
    score = detail["weighted_rmsse"].sum()
    
    return score, detail

In [15]:
def wrmsse_score_by_horizon_split(
    actual_wide: pd.DataFrame,
    pred_wide: pd.DataFrame,
    metric_info: pd.DataFrame
):
    actual_first = actual_wide.iloc[:, :28]
    pred_first = pred_wide.iloc[:, :28]
    
    actual_second = actual_wide.iloc[:, 28:56]
    pred_second = pred_wide.iloc[:, 28:56]
    
    score_first, detail_first = wrmsse_score(actual_first, pred_first, metric_info)
    score_second, detail_second = wrmsse_score(actual_second, pred_second, metric_info)
    
    return score_first, score_second, detail_first, detail_second

In [16]:
folds = [
    {
        "fold": "recent_2025",
        "train_end": "2025-07-11",
        "valid_start": "2025-07-12",
        "horizon": 56
    },
    {
        "fold": "seasonal_2024",
        "train_end": "2024-09-05",
        "valid_start": "2024-09-06",
        "horizon": 56
    },
    {
        "fold": "seasonal_2023",
        "train_end": "2023-09-05",
        "valid_start": "2023-09-06",
        "horizon": 56
    }
]

In [17]:
cv_records = []
detail_outputs = {}

for fold_cfg in folds:
    fold_name = fold_cfg["fold"]
    train_end = fold_cfg["train_end"]
    valid_start = fold_cfg["valid_start"]
    horizon = fold_cfg["horizon"]
    
    print("=" * 80)
    print("Fold:", fold_name)
    
    metric_info_fold = compute_sku_metric_info(
        daily_panel,
        train_end=train_end,
        target_col=TARGET_FOR_METRIC
    )
    
    actual_wide = make_actual_matrix(
        daily_panel,
        start_date=valid_start,
        horizon=horizon,
        target_col=TARGET_FOR_METRIC
    )
    
    pred_baseline_v1 = make_baseline_v1_forecast(
        daily_panel,
        sku_activity=sku_activity,
        train_end=train_end,
        forecast_start=valid_start,
        horizon=horizon,
        target_col=TARGET_FOR_PRED,
        sunday_factor=0.0
    )
    
    score_full, detail_full = wrmsse_score(
        actual_wide,
        pred_baseline_v1,
        metric_info_fold
    )
    
    score_first28, score_second28, _, _ = wrmsse_score_by_horizon_split(
        actual_wide,
        pred_baseline_v1,
        metric_info_fold
    )
    
    cv_records.append({
        "fold": fold_name,
        "model": "baseline_v1",
        "wrmsse_full56": score_full,
        "wrmsse_first28": score_first28,
        "wrmsse_second28": score_second28
    })
    
    detail_outputs[fold_name] = detail_full
    
    print("WRMSSE full 56:", score_full)
    print("WRMSSE first 28:", score_first28)
    print("WRMSSE second 28:", score_second28)

cv_baseline_v1 = pd.DataFrame(cv_records)
display(cv_baseline_v1)

cv_baseline_v1.to_csv(CV_DIR / "step4_baseline_v1_cv.csv", index=False)

Fold: recent_2025
WRMSSE full 56: 0.5892320943666405
WRMSSE first 28: 0.5304228644368046
WRMSSE second 28: 0.5643596962703616
Fold: seasonal_2024
WRMSSE full 56: 0.753451080472963
WRMSSE first 28: 0.6611441848265287
WRMSSE second 28: 0.7262940746721616
Fold: seasonal_2023
WRMSSE full 56: 0.8165782164719191
WRMSSE first 28: 0.6456654384060494
WRMSSE second 28: 0.8347668397028323


,fold,model,wrmsse_full56,wrmsse_first28,wrmsse_second28
0,recent_2025,baseline_v1,0.589232,0.530423,0.564360
1,seasonal_2024,baseline_v1,0.753451,0.661144,0.726294
2,seasonal_2023,baseline_v1,0.816578,0.645665,0.834767


## 10. Diagnostic errors by bucket

In [18]:
def assign_rank_bucket(rank):
    if rank <= 100:
        return "rank_001_100"
    elif rank <= 500:
        return "rank_101_500"
    elif rank <= 1000:
        return "rank_501_1000"
    elif rank <= 2000:
        return "rank_1001_2000"
    else:
        return "rank_2001_plus"

diagnostic_detail = detail_outputs["recent_2025"].copy()
diagnostic_detail["rank_bucket"] = diagnostic_detail["profit_rank"].apply(assign_rank_bucket)

bucket_summary = (
    diagnostic_detail.groupby("rank_bucket", as_index=False)
    .agg(
        sku_count=("ItemCode", "count"),
        weight_sum=("weight", "sum"),
        wrmsse_contribution=("weighted_rmsse", "sum"),
        avg_rmsse=("rmsse", "mean")
    )
    .sort_values("wrmsse_contribution", ascending=False)
)

display(bucket_summary)

bucket_summary.to_csv(CV_DIR / "step4_baseline_v1_recent_2025_bucket_summary.csv", index=False)

display(
    diagnostic_detail
    .sort_values("weighted_rmsse", ascending=False)
    .head(30)
)

,rank_bucket,sku_count,weight_sum,wrmsse_contribution,avg_rmsse
0,rank_001_100,100,0.397870,0.276372,0.496024
2,rank_101_500,400,0.235905,0.138883,0.591187
4,rank_501_1000,500,0.114096,0.062181,0.529765
3,rank_2001_plus,13972,0.146865,0.061456,89.399477
1,rank_1001_2000,1000,0.105264,0.050341,0.471500


,ItemCode,mse,weight,scale,scale_safe,total_profit_metric_train,positive_profit,profit_rank,rmsse,weighted_rmsse,rank_bucket
2,SKU-00003,89.743156,0.095070,56.281674,56.281674,1.587671e+10,1.587671e+10,1,1.262749,0.120049,rank_001_100
1,SKU-00002,28.775618,0.045248,22.822039,22.822039,7.556427e+09,7.556427e+09,2,1.122885,0.050808,rank_001_100
13993,SKU-14323,5314.945763,0.005414,1509.051267,1509.051267,9.041417e+08,9.041417e+08,10,1.876711,0.010161,rank_001_100
15241,SKU-15599,17830.028665,0.002707,3148.621685,3148.621685,4.520825e+08,4.520825e+08,26,2.379665,0.006442,rank_001_100
13990,SKU-14320,2167.502615,0.005120,1623.653506,1623.653506,8.549961e+08,8.549961e+08,11,1.155402,0.005915,rank_001_100
10248,SKU-10532,140.823261,0.002977,70.189747,70.189747,4.972008e+08,4.972008e+08,22,1.416447,0.004217,rank_001_100
11100,SKU-11398,1688.005216,0.000676,47.646435,47.646435,1.128099e+08,1.128099e+08,222,5.952120,0.004021,rank_101_500
8616,SKU-08863,1659.878745,0.003742,1627.361226,1627.361226,6.249812e+08,6.249812e+08,17,1.009941,0.003780,rank_001_100
6593,SKU-06772,183.064712,0.000679,6.265174,6.265174,1.133406e+08,1.133406e+08,220,5.405499,0.003669,rank_101_500
9839,SKU-10117,1791.207998,0.001124,298.143783,298.143783,1.876291e+08,1.876291e+08,114,2.451095,0.002754,rank_101_500


## 11. Build final forecast 56 days

In [19]:
final_pred_56 = make_baseline_v1_forecast(
    daily_panel,
    sku_activity=sku_activity,
    train_end=FINAL_TRAIN_END,
    forecast_start=FINAL_FORECAST_START,
    horizon=HORIZON,
    target_col=TARGET_FOR_PRED,
    sunday_factor=0.0
)

print(final_pred_56.shape)
display(final_pred_56.head())

print("min pred:", final_pred_56.min().min())
print("max pred:", final_pred_56.max().max())
print("mean pred:", final_pred_56.values.mean())
print("total forecast qty:", final_pred_56.values.sum())

(15972, 56)


,2025-09-06,2025-09-07,2025-09-08,2025-09-09,2025-09-10,2025-09-11,2025-09-12,2025-09-13,2025-09-14,2025-09-15,2025-09-16,2025-09-17,2025-09-18,2025-09-19,2025-09-20,2025-09-21,2025-09-22,2025-09-23,2025-09-24,2025-09-25,2025-09-26,2025-09-27,2025-09-28,2025-09-29,2025-09-30,2025-10-01,2025-10-02,2025-10-03,2025-10-04,2025-10-05,2025-10-06,2025-10-07,2025-10-08,2025-10-09,2025-10-10,2025-10-11,2025-10-12,2025-10-13,2025-10-14,2025-10-15,2025-10-16,2025-10-17,2025-10-18,2025-10-19,2025-10-20,2025-10-21,2025-10-22,2025-10-23,2025-10-24,2025-10-25,2025-10-26,2025-10-27,2025-10-28,2025-10-29,2025-10-30,2025-10-31
SKU-00001,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038
SKU-00002,5.404574,0.0,7.106497,6.475728,6.731497,6.854574,6.925728,5.404574,0.0,7.556497,7.450728,6.806497,6.854574,6.550728,5.629574,0.0,7.406497,7.225728,7.856497,7.379574,7.375728,5.629574,0.0,8.531497,7.150728,7.106497,7.079574,6.625728,5.629574,0.0,7.031497,7.600728,6.881497,6.704574,7.300728,5.554574,0.0,7.631497,7.000728,7.406497,7.904574,6.550728,5.704574,0.0,7.106497,7.150728,7.406497,6.929574,7.525728,5.854574,0.0,7.631497,7.600728,7.556497,7.454574,6.700728
SKU-00003,9.784251,0.0,12.884251,11.888098,11.709251,12.790021,10.841944,10.159251,0.0,12.509251,11.738098,12.759251,13.090021,12.191944,10.309251,0.0,13.184251,11.813098,11.784251,13.390021,12.491944,10.609251,0.0,12.809251,10.688098,12.084251,12.265021,10.241944,10.009251,0.0,10.934251,10.988098,11.784251,13.015021,10.916944,10.759251,0.0,11.534251,11.438098,12.159251,12.865021,10.391944,9.934251,0.0,11.909251,10.688098,11.934251,12.790021,10.691944,10.234251,0.0,12.434251,12.563098,12.459251,13.540021,10.391944
SKU-00004,0.000000,0.0,0.000000,0.075000,0.225000,0.075000,0.300000,0.300000,0.0,0.225000,0.075000,0.375000,0.075000,0.375000,0.150000,0.0,0.525000,0.150000,0.300000,0.525000,0.225000,0.525000,0.0,0.525000,0.225000,0.150000,0.225000,0.225000,0.000000,0.0,0.225000,0.000000,0.375000,0.225000,0.300000,0.225000,0.0,0.150000,0.150000,0.450000,0.600000,0.075000,0.075000,0.0,0.225000,0.150000,0.450000,0.075000,0.525000,0.000000,0.0,0.375000,0.300000,0.375000,0.300000,0.150000
SKU-00005,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000


min pred: 0.0
max pred: 868.6418337912088
mean pred: 0.08508228543089634
total forecast qty: 76100.31872252747


## 12. Convert forcast 56 days to Kaggle submission

In [24]:
def make_kaggle_submission(
    pred_56: pd.DataFrame,
    sample: pd.DataFrame,
    forecast_start,
    output_path=None
) -> pd.DataFrame:
    """
    Convert 56-day forecast matrix into Kaggle submission format:
    - <SKU>_validation: day 1-28
    - <SKU>_evaluation: day 29-56
    """
    forecast_start = pd.Timestamp(forecast_start)
    forecast_dates = pd.date_range(forecast_start, periods=56, freq="D")
    
    validation_dates = forecast_dates[:28]
    evaluation_dates = forecast_dates[28:]
    
    f_cols = [f"F{i}" for i in range(1, 29)]
    
    sub = sample.copy()
    sub[f_cols] = sub[f_cols].astype(float)
    
    # Parse id
    parsed = sub["id"].str.rsplit("_", n=1, expand=True)
    sub["ItemCode_tmp"] = parsed[0]
    sub["window_tmp"] = parsed[1]
    
    # Safety
    missing_skus = set(sub["ItemCode_tmp"]) - set(pred_56.index)
    if missing_skus:
        raise ValueError(f"Missing SKUs in prediction: {len(missing_skus)}")
    
    for idx, row in sub.iterrows():
        sku = row["ItemCode_tmp"]
        window = row["window_tmp"]
        
        if window == "validation":
            values = pred_56.loc[sku, validation_dates].to_numpy(dtype=float)
        elif window == "evaluation":
            values = pred_56.loc[sku, evaluation_dates].to_numpy(dtype=float)
        else:
            raise ValueError(f"Unknown window suffix: {window}")
        
        sub.loc[idx, f_cols] = values
    
    sub = sub.drop(columns=["ItemCode_tmp", "window_tmp"])
    
    # Final checks
    assert sub.shape == sample.shape
    assert set(sub["id"]) == set(sample["id"])
    assert sub["id"].nunique() == len(sub)
    
    values = sub[f_cols].to_numpy(dtype=float)
    
    if np.isnan(values).any():
        raise ValueError("Submission contains NaN.")
    
    if np.isinf(values).any():
        raise ValueError("Submission contains inf.")
    
    if (values < 0).any():
        raise ValueError("Submission contains negative values.")
    
    if output_path is not None:
        sub.to_csv(output_path, index=False)
        print("Saved:", output_path)
    
    return sub

In [25]:
submission_path = SUB_DIR / "submission_baseline_v1.csv"

submission_baseline_v1 = make_kaggle_submission(
    final_pred_56,
    sample=sample,
    forecast_start=FINAL_FORECAST_START,
    output_path=submission_path
)

display(submission_baseline_v1.head())
display(submission_baseline_v1.tail())

print(submission_baseline_v1.shape)

Saved: ../outputs/submissions/submission_baseline_v1.csv


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038
1,SKU-00002_validation,5.404574,0.0,7.106497,6.475728,6.731497,6.854574,6.925728,5.404574,0.0,7.556497,7.450728,6.806497,6.854574,6.550728,5.629574,0.0,7.406497,7.225728,7.856497,7.379574,7.375728,5.629574,0.0,8.531497,7.150728,7.106497,7.079574,6.625728
2,SKU-00003_validation,9.784251,0.0,12.884251,11.888098,11.709251,12.790021,10.841944,10.159251,0.0,12.509251,11.738098,12.759251,13.090021,12.191944,10.309251,0.0,13.184251,11.813098,11.784251,13.390021,12.491944,10.609251,0.0,12.809251,10.688098,12.084251,12.265021,10.241944
3,SKU-00004_validation,0.000000,0.0,0.000000,0.075000,0.225000,0.075000,0.300000,0.300000,0.0,0.225000,0.075000,0.375000,0.075000,0.375000,0.150000,0.0,0.525000,0.150000,0.300000,0.525000,0.225000,0.525000,0.0,0.525000,0.225000,0.150000,0.225000,0.225000
4,SKU-00005_validation,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
31939,SKU-16329_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31940,SKU-16330_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31941,SKU-16331_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31942,SKU-16332_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31943,SKU-16333_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.075,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


(31944, 29)


## 13. QA after creating submission

In [26]:
f_cols = [f"F{i}" for i in range(1, 29)]

print("Shape:", submission_baseline_v1.shape)
print("Unique IDs:", submission_baseline_v1["id"].nunique())
print("Missing values:", submission_baseline_v1[f_cols].isna().sum().sum())
print("Negative values:", (submission_baseline_v1[f_cols] < 0).sum().sum())
print("Total forecast:", submission_baseline_v1[f_cols].sum().sum())
print("Max forecast:", submission_baseline_v1[f_cols].max().max())

assert submission_baseline_v1.shape == sample.shape
assert submission_baseline_v1["id"].tolist() == sample["id"].tolist()
assert submission_baseline_v1[f_cols].isna().sum().sum() == 0
assert (submission_baseline_v1[f_cols] < 0).sum().sum() == 0

Shape: (31944, 29)
Unique IDs: 31944
Missing values: 0
Negative values: 0
Total forecast: 76100.31872252749
Max forecast: 868.6418337912088


In [27]:
validation_sunday_f = ["F2", "F9", "F16", "F23"]
evaluation_sunday_f = ["F2", "F9", "F16", "F23"]

validation_rows = submission_baseline_v1["id"].str.endswith("_validation")
evaluation_rows = submission_baseline_v1["id"].str.endswith("_evaluation")

print("Validation Sunday forecast total:")
print(submission_baseline_v1.loc[validation_rows, validation_sunday_f].sum().sum())

print("Evaluation Sunday forecast total:")
print(submission_baseline_v1.loc[evaluation_rows, evaluation_sunday_f].sum().sum())

Validation Sunday forecast total:
0.0
Evaluation Sunday forecast total:
0.0


## 14. Submission A

In [28]:
# Submission A: recent mean 28 + Sunday zero

pred_recent28 = make_recent_blend_prediction(
    daily_panel,
    train_end=FINAL_TRAIN_END,
    forecast_start=FINAL_FORECAST_START,
    horizon=HORIZON,
    target_col=TARGET_FOR_PRED,
    windows=(28,),
    weights=(1.0,)
)

pred_recent28 = postprocess_prediction(
    pred_recent28,
    panel=daily_panel,
    sku_activity=sku_activity,
    train_end=FINAL_TRAIN_END,
    sunday_factor=0.0,
    apply_cap=True
)

print("recent28 total:", pred_recent28.values.sum())
print("recent28 max:", pred_recent28.max().max())
print("recent28 mean:", pred_recent28.values.mean())

sunday_cols = [c for c in pred_recent28.columns if pd.Timestamp(c).dayofweek == 6]
print("Sunday total:", pred_recent28[sunday_cols].sum().sum())

submission_recent28_path = SUB_DIR / "submission_recent28_sunday0.csv"

submission_recent28 = make_kaggle_submission(
    pred_recent28,
    sample=sample,
    forecast_start=FINAL_FORECAST_START,
    output_path=submission_recent28_path
)

display(submission_recent28.head())

recent28 total: 62516.57142857143
recent28 max: 83.5
recent28 mean: 0.06989527591652739
Sunday total: 0.0


In [29]:
f_cols = [f"F{i}" for i in range(1, 29)]

print("Shape:", submission_recent28.shape)
print("Unique IDs:", submission_recent28["id"].nunique())
print("Missing:", submission_recent28[f_cols].isna().sum().sum())
print("Negative:", (submission_recent28[f_cols] < 0).sum().sum())
print("Total:", submission_recent28[f_cols].sum().sum())
print("Max:", submission_recent28[f_cols].max().max())

assert submission_recent28.shape == sample.shape
assert submission_recent28["id"].tolist() == sample["id"].tolist()
assert submission_recent28[f_cols].isna().sum().sum() == 0
assert (submission_recent28[f_cols] < 0).sum().sum() == 0

Shape: (31944, 29)
Unique IDs: 31944
Missing: 0
Negative: 0
Total: 62516.571428571435
Max: 83.5


## 15. Submission B

In [30]:
# Submission B: recent blend + same DOW, no yearly lag

pred_recent = make_recent_blend_prediction(
    daily_panel,
    train_end=FINAL_TRAIN_END,
    forecast_start=FINAL_FORECAST_START,
    horizon=HORIZON,
    target_col=TARGET_FOR_PRED,
    windows=(28, 56, 112),
    weights=(0.50, 0.30, 0.20)
)

pred_samedow = make_same_dow_prediction(
    daily_panel,
    train_end=FINAL_TRAIN_END,
    forecast_start=FINAL_FORECAST_START,
    horizon=HORIZON,
    target_col=TARGET_FOR_PRED,
    window_days=182
)

pred_no_yearly = (
    0.55 * pred_recent +
    0.45 * pred_samedow
)

pred_no_yearly = postprocess_prediction(
    pred_no_yearly,
    panel=daily_panel,
    sku_activity=sku_activity,
    train_end=FINAL_TRAIN_END,
    sunday_factor=0.0,
    apply_cap=True
)

print("no_yearly total:", pred_no_yearly.values.sum())
print("no_yearly max:", pred_no_yearly.max().max())
print("no_yearly mean:", pred_no_yearly.values.mean())

sunday_cols = [c for c in pred_no_yearly.columns if pd.Timestamp(c).dayofweek == 6]
print("Sunday total:", pred_no_yearly[sunday_cols].sum().sum())

submission_no_yearly_path = SUB_DIR / "submission_baseline_no_yearly.csv"

submission_no_yearly = make_kaggle_submission(
    pred_no_yearly,
    sample=sample,
    forecast_start=FINAL_FORECAST_START,
    output_path=submission_no_yearly_path
)

display(submission_no_yearly.head())

no_yearly total: 71715.32571428572
no_yearly max: 158.76352335164836
no_yearly mean: 0.08017974056639937
Sunday total: 0.0
Saved: ../outputs/submissions/submission_baseline_no_yearly.csv


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.137500,0.0,0.172115,0.241346,0.379808,0.206731,0.206731,0.137500,0.0,0.172115,0.241346,0.379808,0.206731,0.206731,0.137500,0.0,0.172115,0.241346,0.379808,0.206731,0.206731,0.137500,0.0,0.172115,0.241346,0.379808,0.206731,0.206731
1,SKU-00002_validation,5.869052,0.0,6.855591,6.820975,6.855591,6.994052,6.820975,5.869052,0.0,6.855591,6.820975,6.855591,6.994052,6.820975,5.869052,0.0,6.855591,6.820975,6.855591,6.994052,6.820975,5.869052,0.0,6.855591,6.820975,6.855591,6.994052,6.820975
2,SKU-00003_validation,11.272205,0.0,11.722205,11.445282,12.172205,12.881820,10.943359,11.272205,0.0,11.722205,11.445282,12.172205,12.881820,10.943359,11.272205,0.0,11.722205,11.445282,12.172205,12.881820,10.943359,11.272205,0.0,11.722205,11.445282,12.172205,12.881820,10.943359
3,SKU-00004_validation,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
4,SKU-00005_validation,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000


## 16. Build recent baseline variant function

In [31]:
def build_recent_baseline_submission(
    name: str,
    windows,
    weights,
    apply_cap: bool = True,
    sunday_factor: float = 0.0
):
    """
    Build recent-window baseline submission.
    Example:
    - windows=(21,), weights=(1.0,)
    - windows=(28, 56), weights=(0.75, 0.25)
    """
    pred = make_recent_blend_prediction(
        daily_panel,
        train_end=FINAL_TRAIN_END,
        forecast_start=FINAL_FORECAST_START,
        horizon=HORIZON,
        target_col=TARGET_FOR_PRED,
        windows=windows,
        weights=weights
    )

    pred = postprocess_prediction(
        pred,
        panel=daily_panel,
        sku_activity=sku_activity,
        train_end=FINAL_TRAIN_END,
        sunday_factor=sunday_factor,
        apply_cap=apply_cap
    )

    output_path = SUB_DIR / f"submission_{name}.csv"

    sub = make_kaggle_submission(
        pred,
        sample=sample,
        forecast_start=FINAL_FORECAST_START,
        output_path=output_path
    )

    f_cols = [f"F{i}" for i in range(1, 29)]
    validation_rows = sub["id"].str.endswith("_validation")
    evaluation_rows = sub["id"].str.endswith("_evaluation")
    sunday_f = ["F2", "F9", "F16", "F23"]

    summary = {
        "name": name,
        "windows": str(windows),
        "weights": str(weights),
        "apply_cap": apply_cap,
        "sunday_factor": sunday_factor,
        "total_submission": sub[f_cols].sum().sum(),
        "validation_total": sub.loc[validation_rows, f_cols].sum().sum(),
        "evaluation_total": sub.loc[evaluation_rows, f_cols].sum().sum(),
        "max_pred": sub[f_cols].max().max(),
        "missing": sub[f_cols].isna().sum().sum(),
        "negative": (sub[f_cols] < 0).sum().sum(),
        "validation_sunday_total": sub.loc[validation_rows, sunday_f].sum().sum(),
        "evaluation_sunday_total": sub.loc[evaluation_rows, sunday_f].sum().sum(),
        "path": str(output_path)
    }

    return pred, sub, summary

Create candidate

In [32]:
candidate_configs = [
    # recent best performer, rebuild with cap and Sunday zero
    {
        "name": "recent28_sunday0_rebuild",
        "windows": (28,),
        "weights": (1.0,),
        "apply_cap": True,
        "sunday_factor": 0.0
    },

    # Follow trend 
    {
        "name": "recent21_sunday0",
        "windows": (21,),
        "weights": (1.0,),
        "apply_cap": True,
        "sunday_factor": 0.0
    },

    # Follow trend very closely, slightly risky
    {
        "name": "recent14_sunday0",
        "windows": (14,),
        "weights": (1.0,),
        "apply_cap": True,
        "sunday_factor": 0.0
    },

    # More stable, but may miss some recent spikes
    {
        "name": "recent28_56blend_sunday0",
        "windows": (28, 56),
        "weights": (0.75, 0.25),
        "apply_cap": True,
        "sunday_factor": 0.0
    },

    # Even more stable, but may miss more recent spikes
    {
        "name": "recent21_28blend_sunday0",
        "windows": (21, 28),
        "weights": (0.50, 0.50),
        "apply_cap": True,
        "sunday_factor": 0.0
    },
]

candidate_preds = {}
candidate_subs = {}
candidate_summaries = []

for cfg in candidate_configs:
    pred, sub, summary = build_recent_baseline_submission(**cfg)
    candidate_preds[cfg["name"]] = pred
    candidate_subs[cfg["name"]] = sub
    candidate_summaries.append(summary)

candidate_summary_df = pd.DataFrame(candidate_summaries)
display(candidate_summary_df)

Saved: ../outputs/submissions/submission_recent28_sunday0_rebuild.csv
Saved: ../outputs/submissions/submission_recent21_sunday0.csv
Saved: ../outputs/submissions/submission_recent14_sunday0.csv
Saved: ../outputs/submissions/submission_recent28_56blend_sunday0.csv
Saved: ../outputs/submissions/submission_recent21_28blend_sunday0.csv


,name,windows,weights,apply_cap,sunday_factor,total_submission,validation_total,evaluation_total,max_pred,missing,negative,validation_sunday_total,evaluation_sunday_total,path
0,recent28_sunday0_rebuild,"(28,)","(1.0,)",True,0.0,62516.571429,31258.285714,31258.285714,83.500000,0,0,0.0,0.0,../outputs/submissions/submission_recent28_sun...
1,recent21_sunday0,"(21,)","(1.0,)",True,0.0,61099.428571,30549.714286,30549.714286,93.714286,0,0,0.0,0.0,../outputs/submissions/submission_recent21_sun...
2,recent14_sunday0,"(14,)","(1.0,)",True,0.0,56064.000000,28032.000000,28032.000000,85.428571,0,0,0.0,0.0,../outputs/submissions/submission_recent14_sun...
3,recent28_56blend_sunday0,"(28, 56)","(0.75, 0.25)",True,0.0,62833.071429,31416.535714,31416.535714,82.750000,0,0,0.0,0.0,../outputs/submissions/submission_recent28_56b...
4,recent21_28blend_sunday0,"(21, 28)","(0.5, 0.5)",True,0.0,61808.000000,30904.000000,30904.000000,88.607143,0,0,0.0,0.0,../outputs/submissions/submission_recent21_28b...
